# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aleezafatima-21/Aleeza-flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
# ML-08: Build warehouse feature frame and March-April decline label

import os
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

# Connect to the FlyRank warehouse
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(
    "CREATE OR REPLACE SECRET hf_secret "
    "(TYPE HUGGINGFACE, TOKEN ?)",
    [os.environ["HF_TOKEN"]]
)

# Build March feature frame
# client_hash_id is retained because the split is grouped by client.
feature_frame = con.execute("""
    SELECT
        content_hash_id,
        ANY_VALUE(client_hash_id) AS client_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(ga4_sessions) AS ga4_sessions,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/'
        'fact_content_daily_performance/month=2026-03/*.parquet'
    )
    GROUP BY content_hash_id
""").df()

print("Feature frame shape:", feature_frame.shape)
print("Unique content:", feature_frame["content_hash_id"].nunique())
print("Unique clients:", feature_frame["client_hash_id"].nunique())

# Build March-April decline label
label_data = con.execute("""
    WITH march AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS march_impressions
        FROM read_parquet(
            'hf://datasets/FlyRank/internship-warehouse/'
            'fact_content_daily_performance/month=2026-03/*.parquet'
        )
        GROUP BY content_hash_id
    ),
    april AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS april_impressions
        FROM read_parquet(
            'hf://datasets/FlyRank/internship-warehouse/'
            'fact_content_daily_performance/month=2026-04/*.parquet'
        )
        GROUP BY content_hash_id
    )
    SELECT
        march.content_hash_id,
        march.march_impressions,
        april.april_impressions,
        CASE
            WHEN april.april_impressions < march.march_impressions
            THEN 1
            ELSE 0
        END AS is_declining_label
    FROM march
    INNER JOIN april
        ON march.content_hash_id = april.content_hash_id
""").df()

print("\nLabel rows:", len(label_data))

print("\nLabel distribution:")
print(label_data["is_declining_label"].value_counts())

# Merge features and labels
data = feature_frame.merge(
    label_data[["content_hash_id", "is_declining_label"]],
    on="content_hash_id",
    how="inner"
)

# Make row order deterministic before splitting/model training
data = data.sort_values("content_hash_id").reset_index(drop=True)

print("\nRows:", len(data))
print("Clients:", data["client_hash_id"].nunique())

print("\nMissing values:")
print(data.isna().sum())

# Check the special GSC position values
print(
    "\ngsc_avg_position == 0:",
    (data["gsc_avg_position"] == 0).sum()
)

print(
    "gsc_avg_position missing:",
    data["gsc_avg_position"].isna().sum()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (331437, 7)
Unique content: 331437
Unique clients: 55


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Label rows: 331436

Label distribution:
is_declining_label
0    219469
1    111967
Name: count, dtype: int64

Rows: 331436
Clients: 55

Missing values:
content_hash_id              0
client_hash_id               0
gsc_impressions              0
gsc_clicks                   0
gsc_avg_position        154699
ga4_sessions             70700
ga4_engaged_sessions     70700
is_declining_label           0
dtype: int64

gsc_avg_position == 0: 1434
gsc_avg_position missing: 154699


In [3]:
features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(data, groups=data["client_hash_id"]))
train = data.iloc[train_idx].copy()
test = data.iloc[test_idx].copy()

In [4]:
train_medians = train[features].copy()
train_medians["gsc_avg_position"] = train_medians["gsc_avg_position"].replace(0, np.nan)
train_medians = train_medians.median()

def prep(d):
    X = d[features].copy()
    X["gsc_avg_position"] = X["gsc_avg_position"].replace(0, np.nan)
    return X.fillna(train_medians)

X_train = prep(train)
X_test = prep(test)
y_train = train["is_declining_label"]
y_test = test["is_declining_label"]

In [5]:
from sklearn.metrics import precision_score

def precision_at_k(y_true, scores, k=50, tie_break=None):
    scores = np.asarray(scores, dtype=float)
    if tie_break is not None:
        tie_break = np.asarray(tie_break, dtype=float)
        order = np.lexsort((-tie_break, -scores))
    else:
        order = np.argsort(-scores, kind="stable")
    top_k = order[:k]
    return precision_score(np.asarray(y_true)[top_k], np.ones(len(top_k)), zero_division=0)

In [6]:
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

In [7]:
# ML-09: Naive random split vs. client-grouped split (before/after)

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# --- "BEFORE": naive random 80/20 split, ignoring client_hash_id ---
train_naive, test_naive = train_test_split(
    data, test_size=0.20, random_state=42
)

def prep_naive(d, medians):
    X = d[features].copy()
    X["gsc_avg_position"] = X["gsc_avg_position"].replace(0, np.nan)
    return X.fillna(medians)

naive_medians = train_naive[features].median()
X_train_naive = prep_naive(train_naive, naive_medians)
X_test_naive = prep_naive(test_naive, naive_medians)
y_train_naive = train_naive["is_declining_label"]
y_test_naive = test_naive["is_declining_label"]

# Check for client overlap (this is the leak we expect to find)
overlap_naive = set(train_naive["client_hash_id"]) & set(test_naive["client_hash_id"])
print("Naive split — client overlap:", len(overlap_naive), "of", data["client_hash_id"].nunique(), "clients")

rf_naive = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_naive.fit(X_train_naive, y_train_naive)
naive_proba = rf_naive.predict_proba(X_test_naive)[:, 1]

print("\n--- BEFORE: naive random split ---")
print("ROC-AUC:", round(roc_auc_score(y_test_naive, naive_proba), 3))
print("Average Precision:", round(average_precision_score(y_test_naive, naive_proba), 3))
print("Precision@50:", round(precision_at_k(y_test_naive, naive_proba, k=50), 3))

print("\n--- AFTER: client-grouped split (from w05) ---")
print("ROC-AUC:", round(roc_auc_score(y_test, rf_proba), 3))
print("Average Precision:", round(average_precision_score(y_test, rf_proba), 3))
print("Precision@50:", round(precision_at_k(y_test, rf_proba, k=50), 3))

Naive split — client overlap: 55 of 55 clients

--- BEFORE: naive random split ---
ROC-AUC: 0.887
Average Precision: 0.726
Precision@50: 0.88

--- AFTER: client-grouped split (from w05) ---
ROC-AUC: 0.851
Average Precision: 0.654
Precision@50: 0.74


### Section 2 — Naive split vs. client-grouped split

To check whether splitting the data randomly could give an overly optimistic estimate of model performance, I retrained the random forest using a naive random 80/20 split that ignored client identity. In this split, all 55 clients appeared in both the training and test sets.

The naive split gave a ROC-AUC of **0.887**, average precision of **0.726**, and Precision@50 of **0.88**. These results were higher than the client-grouped split, which gave a ROC-AUC of **0.851**, average precision of **0.654**, and Precision@50 of **0.74**.

The performance drop after grouping by client suggests that the naive split was giving the model an easier test set because content from the same clients could appear in both training and test data. This means the naive split may partly benefit from client-specific patterns that would not be available when making predictions for a completely new client.

For this reason, I consider the client-grouped split to be a more realistic and honest estimate of how the model would generalize to unseen clients. Therefore, I use the client-grouped results — **ROC-AUC 0.851 and Precision@50 0.74** — for the rest of the project.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [8]:
# ML-09: Quick leakage sanity check — is GA4 missingness itself predictive?

ga4_missing = data["ga4_sessions"].isna()
print("Decline rate when GA4 missing:", round(data.loc[ga4_missing, "is_declining_label"].mean(), 3))
print("Decline rate when GA4 present:", round(data.loc[~ga4_missing, "is_declining_label"].mean(), 3))

Decline rate when GA4 missing: 0.41
Decline rate when GA4 present: 0.318


### Section 3 — Leakage audit

I checked the data and feature pipeline for possible sources of leakage. The label was created using March and April 2026 GSC impressions, so information from later months was not used to define the target. I also used a client-grouped split, as discussed in Section 2, to prevent content from the same client appearing in both the training and test sets. The difference between the naive split (ROC-AUC 0.887) and the grouped split (ROC-AUC 0.851) showed why the grouped split gives a more realistic estimate of model performance.

I also checked whether missing GA4 data was related to the decline label. Pages with missing `ga4_sessions` had a **41.0% decline rate**, compared with **31.8%** for pages where GA4 data was present. This is not leakage because the missingness does not use future information or information from the test set. However, it shows that missing GA4 data contains useful information about the pages. By replacing missing values with the training median, the current pipeline treats missing data similarly to a real value of zero, which can hide this difference.

The `ga4_data_available` flag was not included as a feature in the current experiment. A future version could include this flag explicitly so the model can distinguish between missing GA4 data and actual zero activity.

Overall, I did not identify any other major leakage issues. The features were built from the intended March data, and preprocessing values such as medians and thresholds were calculated using the training data before being applied to the test data.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Section 4 — Claim rewrite

On the held-out client split, the random forest achieved the highest ROC-AUC, average precision, F1, and Precision@50 among the models I tested. However, these results come from one client-grouped train/test split of 55 clients and one March–April 2026 label period. Therefore, they do not prove that the random forest will always perform better on new clients, different time periods, or production data.

Based on these results, the random forest is a reasonable model to use as a decision-support tool for prioritizing pages for review. I would treat its performance as evidence from this experiment rather than a guarantee that its rankings will remain accurate in the future.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.